<a href="https://colab.research.google.com/github/treborskrub/Three_Agent_Process_Engine_v0_4_1/blob/main/Three_Agent_Process_Engine_v0_4_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# =============================================================================
# THREE-AGENT ORCHESTRATED PROCESS ENGINE v0.4.1
# SAFE PATH-LEVEL ADAPTIVE PRUNING
# =============================================================================
# Solid working baseline:
# - Balance Agent: closer / farther from SetState
# - Efficiency Agent: productive / wasteful / stalled
# - Shortfall Agent: BALANCE / WATCH / SHORTFALL_1_3 / CRITICAL / EFFICIENCY
# - Orchestrator: retain, monitor, prune, restore, protect, or reseed
# - Safety Guard: blocks a prune that would drop below the safety floor
#
# This is an abstract, deterministic process-control testbed.
# It is not a physical, semantic-reasoning, or quantum-hardware simulation.
# =============================================================================

from dataclasses import dataclass
from copy import deepcopy

EPSILON = 1e-9

# -----------------------------------------------------------------------------
# CONTROL PANEL
# These are the main values to adjust in future experiments.
# -----------------------------------------------------------------------------
SETSTATE_TASK_SUM = 100.0
MAX_CYCLES = 12

BALANCE_FLOOR = 0.90          # Normal balance begins at 90% of SetState
WATCH_FLOOR = 0.80            # Watch range begins at 80%
CRITICAL_FLOOR = 0.67         # Critical state begins below 67%
CRITICAL_EXIT_RATIO = 0.80    # Lockout ends only after recovery reaches 80%

WASTEFUL_CONFIRMATION_CYCLES = 2
MAX_CRITICAL_ENTRIES = 3

# -----------------------------------------------------------------------------
# PATH MODEL
# -----------------------------------------------------------------------------
@dataclass
class TaskPath:
    name: str
    contribution: float
    cost: float
    reliability: float
    priority: str
    protected: bool = False
    active: bool = True
    status: str = "ACTIVE"
    monitored_cycles: int = 0

    def value_score(self):
        """Higher score means better contribution per cost, adjusted for reliability."""
        return (self.contribution * self.reliability) / max(self.cost, EPSILON)

# -----------------------------------------------------------------------------
# AGENT 1: BALANCE
# -----------------------------------------------------------------------------
def balance_agent(setstate, previous_sum, current_sum):
    ratio = current_sum / max(setstate, EPSILON)
    previous_ratio = previous_sum / max(setstate, EPSILON)

    distance = abs(1.0 - ratio)
    previous_distance = abs(1.0 - previous_ratio)

    if distance < previous_distance:
        direction = "CLOSER"
    elif distance > previous_distance:
        direction = "FARTHER"
    else:
        direction = "NO_CHANGE"

    if BALANCE_FLOOR <= ratio < 1.05:
        status = "IN_BALANCE"
    elif WATCH_FLOOR <= ratio < BALANCE_FLOOR:
        status = "DRIFTING"
    elif ratio >= 1.05:
        status = "EXCESS"
    else:
        status = "OUT_OF_BALANCE"

    return {
        "ratio": ratio,
        "distance": distance,
        "direction": direction,
        "status": status,
    }

# -----------------------------------------------------------------------------
# AGENT 2: EFFICIENCY
# -----------------------------------------------------------------------------
def efficiency_agent(setstate, previous_sum, current_sum, previous_cost):
    previous_distance = abs(1.0 - previous_sum / max(setstate, EPSILON))
    current_distance = abs(1.0 - current_sum / max(setstate, EPSILON))

    improvement = previous_distance - current_distance
    efficiency = improvement / max(previous_cost, EPSILON)

    if efficiency > 0.0001:
        status = "PRODUCTIVE"
    elif efficiency < -0.0001:
        status = "WASTEFUL"
    else:
        status = "STALLED"

    return {
        "improvement": improvement,
        "efficiency": efficiency,
        "status": status,
    }

# -----------------------------------------------------------------------------
# AGENT 3: SHORTFALL
# -----------------------------------------------------------------------------
def shortfall_agent(setstate, current_sum):
    ratio = current_sum / max(setstate, EPSILON)

    if ratio < CRITICAL_FLOOR:
        state = "CRITICAL"
    elif ratio < WATCH_FLOOR:
        state = "SHORTFALL_1_3"
    elif ratio < BALANCE_FLOOR:
        state = "WATCH"
    elif ratio >= 1.05:
        state = "EFFICIENCY"
    else:
        state = "BALANCE"

    return {
        "ratio": ratio,
        "state": state,
    }

# -----------------------------------------------------------------------------
# PATH HELPERS
# -----------------------------------------------------------------------------
def active_task_sum(paths):
    return sum(path.contribution for path in paths if path.active)

def active_cost(paths):
    return sum(path.cost for path in paths if path.active)

def priority_rank(priority):
    return {
        "CORE": 3,
        "HIGH": 2,
        "NORMAL": 1,
        "LOW": 0,
    }.get(priority, 0)

def get_prune_candidates(paths):
    candidates = [
        path for path in paths
        if path.active and not path.protected and path.priority != "CORE"
    ]

    candidates.sort(
        key=lambda path: (
            priority_rank(path.priority),
            path.value_score(),
            path.reliability,
        )
    )
    return candidates

def get_restore_candidates(paths):
    candidates = [
        path for path in paths
        if not path.active and not path.protected
    ]

    candidates.sort(
        key=lambda path: (
            -priority_rank(path.priority),
            -path.value_score(),
            -path.reliability,
        )
    )
    return candidates

# -----------------------------------------------------------------------------
# SAFE PRUNING OPERATION
# First observation: mark low-value path as MONITORED.
# Later confirmation: prune only if projected sum remains above safety floor.
# -----------------------------------------------------------------------------
def safe_prune_or_monitor(paths, setstate, force_prune=False):
    candidates = get_prune_candidates(paths)

    if not candidates:
        return {
            "changed_paths": [],
            "result": "NO_OPTIONAL_PATH_AVAILABLE",
            "projected_ratio": active_task_sum(paths) / max(setstate, EPSILON),
        }

    candidate = candidates[0]
    current_sum = active_task_sum(paths)
    projected_sum = current_sum - candidate.contribution
    projected_ratio = projected_sum / max(setstate, EPSILON)

    if not force_prune and candidate.monitored_cycles < 1:
        candidate.monitored_cycles += 1
        candidate.status = "MONITORED"

        return {
            "changed_paths": [candidate.name],
            "result": "MONITORED_NOT_PRUNED",
            "projected_ratio": projected_ratio,
        }

    if projected_ratio < BALANCE_FLOOR:
        candidate.monitored_cycles += 1
        candidate.status = "PRUNE_BLOCKED_BY_SAFETY_GUARD"

        return {
            "changed_paths": [candidate.name],
            "result": "PRUNE_BLOCKED_BY_SAFETY_GUARD",
            "projected_ratio": projected_ratio,
        }

    candidate.active = False
    candidate.status = "PRUNED"
    candidate.monitored_cycles = 0

    return {
        "changed_paths": [candidate.name],
        "result": "PRUNED_SAFELY",
        "projected_ratio": projected_ratio,
    }

# -----------------------------------------------------------------------------
# RECOVERY AND PROTECTION OPERATIONS
# -----------------------------------------------------------------------------
def restore_one_best_path(paths):
    candidates = get_restore_candidates(paths)

    if not candidates:
        return []

    path = candidates[0]
    path.active = True
    path.status = "RESTORED"
    path.monitored_cycles = 0
    return [path.name]

def protect_core_and_restore(paths):
    changed = []

    for path in paths:
        if path.protected or path.priority == "CORE":
            if not path.active:
                path.active = True
                path.status = "CORE_RESTORED"
                changed.append(path.name)
            else:
                path.status = "CORE_PROTECTED"

    optional_active = [
        path for path in paths
        if path.active and not path.protected and path.priority != "CORE"
    ]

    optional_active.sort(key=lambda path: path.value_score())

    for path in optional_active[:1]:
        path.active = False
        path.status = "PROTECTIVE_PRUNE"
        path.monitored_cycles = 0
        changed.append(path.name)

    return changed

def reseed_from_protected_source(paths):
    changed = []

    for path in paths:
        if path.protected or path.priority == "CORE":
            if not path.active:
                path.active = True
                changed.append(path.name)
            path.status = "SOURCE_RESEEDED"
        else:
            if path.active:
                changed.append(path.name)
            path.active = False
            path.status = "RESET_PRUNED"
            path.monitored_cycles = 0

    return changed

# -----------------------------------------------------------------------------
# ORCHESTRATOR
# -----------------------------------------------------------------------------
def orchestrator(balance, efficiency, shortfall,
                 critical_lockout, critical_entries, wasteful_cycles):
    ratio = shortfall["ratio"]
    state = shortfall["state"]

    if critical_entries >= MAX_CRITICAL_ENTRIES:
        return {
            "state": "RESET_REQUIRED",
            "action": "RESEED_FROM_PROTECTED_SOURCE",
            "reason": "Repeated critical entries exceeded the safe recovery limit.",
            "critical_lockout": True,
        }

    if critical_lockout and ratio < CRITICAL_EXIT_RATIO:
        return {
            "state": "CRITICAL_LOCKOUT",
            "action": "PROTECT_CORE_AND_RESTORE",
            "reason": (
                f"Protected recovery remains active until ratio reaches "
                f"{CRITICAL_EXIT_RATIO:.2f}."
            ),
            "critical_lockout": True,
        }

    if state == "CRITICAL":
        return {
            "state": "CRITICAL",
            "action": "PROTECT_CORE_AND_RESTORE",
            "reason": "Task is below 67% of SetState; protected recovery activated.",
            "critical_lockout": True,
        }

    if state == "SHORTFALL_1_3":
        return {
            "state": "SHORTFALL_1_3",
            "action": "SAFE_PRUNE_OR_MONITOR",
            "reason": "Task is between 67% and 80% of SetState.",
            "critical_lockout": False,
        }

    if state == "WATCH":
        if balance["direction"] == "CLOSER":
            return {
                "state": "WATCH",
                "action": "RESTORE_ONE_BEST_PATH",
                "reason": "Recovery is improving; cautiously restore one worthwhile path.",
                "critical_lockout": False,
            }

        return {
            "state": "WATCH",
            "action": "SAFE_PRUNE_OR_MONITOR",
            "reason": "Task is in watch range without confirmed improvement.",
            "critical_lockout": False,
        }

    if state == "EFFICIENCY":
        return {
            "state": "EFFICIENCY",
            "action": "SAFE_PRUNE_OR_MONITOR",
            "reason": "Task exceeds 105% of SetState; inspect excess activity safely.",
            "critical_lockout": False,
        }

    if efficiency["status"] == "WASTEFUL":
        if wasteful_cycles >= WASTEFUL_CONFIRMATION_CYCLES:
            return {
                "state": "BALANCE",
                "action": "SAFE_PRUNE_OR_MONITOR",
                "reason": (
                    "Repeated wasteful movement confirmed; inspect the lowest-value "
                    "optional path with safety protection."
                ),
                "critical_lockout": False,
            }

        return {
            "state": "BALANCE",
            "action": "HOLD_AND_MONITOR",
            "reason": "One wasteful cycle is not enough to authorize pruning.",
            "critical_lockout": False,
        }

    return {
        "state": "BALANCE",
        "action": "RETAIN_PRODUCTIVE_PATHS",
        "reason": "Task is in balance range with no confirmed need for correction.",
        "critical_lockout": False,
    }

# -----------------------------------------------------------------------------
# APPLY ACTION
# -----------------------------------------------------------------------------
def apply_action(paths, action, setstate):
    if action == "SAFE_PRUNE_OR_MONITOR":
        return safe_prune_or_monitor(paths, setstate)

    if action == "RESTORE_ONE_BEST_PATH":
        changed = restore_one_best_path(paths)
        return {
            "changed_paths": changed,
            "result": "PATH_RESTORED" if changed else "NO_PATH_AVAILABLE_TO_RESTORE",
            "projected_ratio": active_task_sum(paths) / max(setstate, EPSILON),
        }

    if action == "PROTECT_CORE_AND_RESTORE":
        changed = protect_core_and_restore(paths)
        return {
            "changed_paths": changed,
            "result": "CORE_PROTECTED_AND_OPTIONALS_REDUCED",
            "projected_ratio": active_task_sum(paths) / max(setstate, EPSILON),
        }

    if action == "RESEED_FROM_PROTECTED_SOURCE":
        changed = reseed_from_protected_source(paths)
        return {
            "changed_paths": changed,
            "result": "RESEEDED_FROM_PROTECTED_SOURCE",
            "projected_ratio": active_task_sum(paths) / max(setstate, EPSILON),
        }

    return {
        "changed_paths": [],
        "result": "NO_PATH_CHANGE",
        "projected_ratio": active_task_sum(paths) / max(setstate, EPSILON),
    }

# -----------------------------------------------------------------------------
# CONTROLLED PATH ENVIRONMENT
# Simulates measured path behavior. This is intentionally deterministic.
# Later, geometry can replace this with real node/path interaction.
# -----------------------------------------------------------------------------
def update_active_path_contributions(paths, cycle):
    for path in paths:
        if not path.active:
            continue

        if path.protected:
            change = 0.5
        elif path.reliability >= 0.85:
            change = 1.0
        elif path.reliability >= 0.60:
            change = 0.2
        else:
            change = -1.5

        if cycle % 4 == 0 and not path.protected:
            change -= 0.5

        path.contribution = max(0.0, path.contribution + change)

# -----------------------------------------------------------------------------
# DISPLAY HELPERS
# -----------------------------------------------------------------------------
def print_paths(paths):
    print(
        f"{'Path':<16}"
        f"{'Contribution':<15}"
        f"{'Cost':<9}"
        f"{'Reliable':<10}"
        f"{'Priority':<10}"
        f"{'Protected':<11}"
        f"{'Active':<8}"
        f"{'Status'}"
    )
    print("-" * 105)

    for path in paths:
        print(
            f"{path.name:<16}"
            f"{path.contribution:<15.2f}"
            f"{path.cost:<9.2f}"
            f"{path.reliability:<10.2f}"
            f"{path.priority:<10}"
            f"{str(path.protected):<11}"
            f"{str(path.active):<8}"
            f"{path.status}"
        )

# -----------------------------------------------------------------------------
# INITIAL PATH CONFIGURATION
# -----------------------------------------------------------------------------
INITIAL_PATHS = [
    TaskPath("Core_Signal",   32.0, 2.0, 0.98, "CORE",   True),
    TaskPath("Core_Memory",   24.0, 2.5, 0.92, "CORE",   True),
    TaskPath("Reliable_Path", 18.0, 2.0, 0.88, "HIGH",   False),
    TaskPath("Medium_Path",   14.0, 4.0, 0.62, "NORMAL", False),
    TaskPath("Weak_Path",     10.0, 6.0, 0.32, "LOW",    False),
]

# -----------------------------------------------------------------------------
# MAIN RUN
# -----------------------------------------------------------------------------
paths = deepcopy(INITIAL_PATHS)

previous_sum = active_task_sum(paths)
previous_cost = max(active_cost(paths), 1.0)

critical_lockout = False
critical_entries = 0
wasteful_cycles = 0
history = []

print("\n" + "=" * 105)
print("THREE-AGENT ORCHESTRATED PROCESS ENGINE v0.4.1")
print("SAFE PATH-LEVEL ADAPTIVE PRUNING")
print("=" * 105)
print(f"SetState Task Sum: {SETSTATE_TASK_SUM:.2f}")
print(f"Balance Safety Floor: {BALANCE_FLOOR:.2f}")
print(f"Initial Active Task Sum: {previous_sum:.2f}")
print("\nINITIAL PATH CONFIGURATION")
print_paths(paths)
print("=" * 105)

for cycle in range(1, MAX_CYCLES + 1):
    current_sum = active_task_sum(paths)
    current_cost = max(active_cost(paths), 1.0)

    balance = balance_agent(SETSTATE_TASK_SUM, previous_sum, current_sum)
    efficiency = efficiency_agent(
        SETSTATE_TASK_SUM, previous_sum, current_sum, previous_cost
    )
    shortfall = shortfall_agent(SETSTATE_TASK_SUM, current_sum)

    if efficiency["status"] == "WASTEFUL":
        wasteful_cycles += 1
    else:
        wasteful_cycles = 0

    if shortfall["state"] == "CRITICAL":
        critical_entries += 1

    decision = orchestrator(
        balance=balance,
        efficiency=efficiency,
        shortfall=shortfall,
        critical_lockout=critical_lockout,
        critical_entries=critical_entries,
        wasteful_cycles=wasteful_cycles,
    )

    critical_lockout = decision["critical_lockout"]

    action_result = apply_action(
        paths=paths,
        action=decision["action"],
        setstate=SETSTATE_TASK_SUM,
    )

    update_active_path_contributions(paths, cycle)
    next_sum = active_task_sum(paths)

    history.append({
        "cycle": cycle,
        "current_sum": current_sum,
        "ratio": balance["ratio"],
        "direction": balance["direction"],
        "efficiency": efficiency["status"],
        "wasteful_cycles": wasteful_cycles,
        "state": decision["state"],
        "action": decision["action"],
        "path_result": action_result["result"],
        "changed_paths": action_result["changed_paths"],
        "next_sum": next_sum,
    })

    print("\n" + "-" * 105)
    print(f"CYCLE {cycle}")
    print("-" * 105)
    print(
        f"Current Sum: {current_sum:.2f} | "
        f"Ratio r: {balance['ratio']:.3f} | "
        f"Direction: {balance['direction']} | "
        f"Balance: {balance['status']}"
    )
    print(
        f"Efficiency: {efficiency['status']} | "
        f"Consecutive Wasteful Cycles: {wasteful_cycles} | "
        f"Shortfall State: {shortfall['state']}"
    )
    print(f"Orchestrator State: {decision['state']}")
    print(f"Action: {decision['action']}")
    print(f"Reason: {decision['reason']}")
    print(f"Path Result: {action_result['result']}")
    print(f"Projected Ratio Before Update: {action_result['projected_ratio']:.3f}")

    if action_result["changed_paths"]:
        print("Changed Paths: " + ", ".join(action_result["changed_paths"]))
    else:
        print("Changed Paths: None")

    print(f"Next-Cycle Task Sum After Path Updates: {next_sum:.2f}")

    previous_sum = current_sum
    previous_cost = current_cost

# -----------------------------------------------------------------------------
# FINAL REPORT
# -----------------------------------------------------------------------------
final_sum = active_task_sum(paths)
final_ratio = final_sum / SETSTATE_TASK_SUM
final_distance = abs(1.0 - final_ratio)

print("\n" + "=" * 105)
print("FINAL PATH CONFIGURATION")
print("=" * 105)
print_paths(paths)

print("\n" + "=" * 105)
print("FINAL CYCLE SUMMARY")
print("=" * 105)
print(
    f"{'Cycle':<7}"
    f"{'Current':<11}"
    f"{'Ratio':<9}"
    f"{'Direction':<12}"
    f"{'Efficiency':<13}"
    f"{'State':<18}"
    f"{'Action':<28}"
    f"{'Path Result':<32}"
    f"{'Next'}"
)
print("-" * 105)

for row in history:
    print(
        f"{row['cycle']:<7}"
        f"{row['current_sum']:<11.2f}"
        f"{row['ratio']:<9.3f}"
        f"{row['direction']:<12}"
        f"{row['efficiency']:<13}"
        f"{row['state']:<18}"
        f"{row['action']:<28}"
        f"{row['path_result']:<32}"
        f"{row['next_sum']:<10.2f}"
    )

print("=" * 105)
print(f"Final Active Task Sum: {final_sum:.2f}")
print(f"Final Ratio r: {final_ratio:.3f}")
print(f"Distance from SetState: {final_distance:.3f}")

if BALANCE_FLOOR <= final_ratio < 1.05:
    verdict = "BALANCE RANGE REACHED"
elif WATCH_FLOOR <= final_ratio < BALANCE_FLOOR:
    verdict = "WATCH / PARTIAL RECOVERY"
elif final_ratio < CRITICAL_FLOOR:
    verdict = "CRITICAL SHORTFALL REMAINS"
elif final_ratio < WATCH_FLOOR:
    verdict = "ONE-THIRD SHORTFALL REMAINS"
else:
    verdict = "EFFICIENCY CORRECTION STILL NEEDED"

print(f"FINAL VERDICT: {verdict}")
print("=" * 105)
print("Saved baseline: v0.4.1 adds projected-impact safety to adaptive pruning.")
print("Later geometry can supply real path contributions and connection effects.")
print("=" * 105)


THREE-AGENT ORCHESTRATED PROCESS ENGINE v0.4.1
SAFE PATH-LEVEL ADAPTIVE PRUNING
SetState Task Sum: 100.00
Balance Safety Floor: 0.90
Initial Active Task Sum: 98.00

INITIAL PATH CONFIGURATION
Path            Contribution   Cost     Reliable  Priority  Protected  Active  Status
---------------------------------------------------------------------------------------------------------
Core_Signal     32.00          2.00     0.98      CORE      True       True    ACTIVE
Core_Memory     24.00          2.50     0.92      CORE      True       True    ACTIVE
Reliable_Path   18.00          2.00     0.88      HIGH      False      True    ACTIVE
Medium_Path     14.00          4.00     0.62      NORMAL    False      True    ACTIVE
Weak_Path       10.00          6.00     0.32      LOW       False      True    ACTIVE

---------------------------------------------------------------------------------------------------------
CYCLE 1
----------------------------------------------------------------------